# Car Racing — PPO with PyTorch

**Environment:** `CarRacing-v3` (top-down, continuous)  
**Algorithm:** Proximal Policy Optimization (PPO-Clip) with CNN actor-critic  

| Property | Detail |
|----------|--------|
| Observation | `(96, 96, 3)` RGB pixel frame |
| Action | `(3,)` continuous — steering · gas · brake |
| Reward | +1000/N tiles visited − 0.1/frame |

## 1. Imports

In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

print(f"Gymnasium : {gym.__version__}")
print(f"PyTorch   : {torch.__version__}")
DEVICE = torch.device("mps" if torch.backends.mps.is_available()
                       else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Device    : {DEVICE}")

## 2. CNN Actor & Critic Networks

The observation is a raw **96×96 RGB** image, so both networks share a CNN backbone.

| Layer | Output shape | Notes |
|-------|-------------|-------|
| Conv(3→32, 8×8, s=4) | 23×23×32 | |
| Conv(32→64, 4×4, s=2) | 10×10×64 | |
| Conv(64→64, 3×3, s=1) | 8×8×64 | |
| Flatten → FC 512 | 512 | shared feature |
| Actor head | μ ∈ (3,), log σ ∈ (3,) | Gaussian policy |
| Critic head | scalar V(s) | |

In [ ]:
N_ACTIONS  = 3
HIDDEN     = 512
LOG_STD_MIN, LOG_STD_MAX = -5, 2


class CNNBase(nn.Module):
    """Shared CNN feature extractor for 96×96 RGB observations."""

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=8, stride=4),   # → (23,23)
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),  # → (10,10)
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),  # → (8,8)
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, HIDDEN),
            nn.ReLU(),
        )
        self.out_dim = HIDDEN

    def forward(self, obs):
        """obs: (B,H,W,C) uint8 or (H,W,C) → normalised feature vector."""
        if obs.dim() == 3:
            obs = obs.unsqueeze(0)
        x = obs.permute(0, 3, 1, 2).float() / 255.0
        return self.net(x)


class ContinuousActor(nn.Module):
    """Gaussian policy: mean (tanh-bounded) + learned log_std per action."""

    def __init__(self, n_actions=N_ACTIONS, lr=3e-4):
        super().__init__()
        self.backbone     = CNNBase()
        self.mean_head    = nn.Linear(HIDDEN, n_actions)
        self.log_std_head = nn.Linear(HIDDEN, n_actions)
        self.optimizer    = optim.Adam(self.parameters(), lr=lr)

    def forward(self, obs):
        feat    = self.backbone(obs)
        mean    = torch.tanh(self.mean_head(feat))
        log_std = self.log_std_head(feat).clamp(LOG_STD_MIN, LOG_STD_MAX)
        return mean, log_std

    @torch.no_grad()
    def select_action(self, obs_np):
        """Sample action and return (action_np, detached log_prob)."""
        obs_t = torch.tensor(obs_np, dtype=torch.uint8).to(DEVICE)
        mean, log_std = self.forward(obs_t)
        std  = log_std.exp()
        dist = torch.distributions.Normal(mean, std)
        raw  = dist.sample()
        act  = torch.tanh(raw)
        # Tanh-squashing log-prob correction
        log_prob = (dist.log_prob(raw) - torch.log(1 - act.pow(2) + 1e-6)).sum(-1)
        return act.squeeze(0).cpu().numpy(), log_prob.detach()

    def evaluate(self, obs_t, actions_t):
        mean, log_std = self.forward(obs_t)
        std  = log_std.exp()
        dist = torch.distributions.Normal(mean, std)
        raw  = torch.atanh(actions_t.clamp(-1 + 1e-6, 1 - 1e-6))
        log_prob = (dist.log_prob(raw) - torch.log(1 - actions_t.pow(2) + 1e-6)).sum(-1)
        entropy  = dist.entropy().sum(-1)
        return log_prob, entropy

    def learn(self, obs_t, actions_t, old_log_probs, advantages,
              epsilon=0.2, entropy_coef=0.01):
        new_log_probs, entropy = self.evaluate(obs_t, actions_t)
        ratio  = torch.exp(new_log_probs - old_log_probs)
        surr1  = ratio * advantages
        surr2  = torch.clamp(ratio, 1 - epsilon, 1 + epsilon) * advantages
        loss   = -torch.min(surr1, surr2).mean() - entropy_coef * entropy.mean()
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.parameters(), 0.5)
        self.optimizer.step()
        return loss.item()


class ContinuousCritic(nn.Module):
    """Value network V(s) for pixel observations."""

    def __init__(self, lr=3e-4):
        super().__init__()
        self.backbone   = CNNBase()
        self.value_head = nn.Linear(HIDDEN, 1)
        self.optimizer  = optim.Adam(self.parameters(), lr=lr)

    def forward(self, obs):
        return self.value_head(self.backbone(obs)).squeeze(-1)

    @torch.no_grad()
    def value(self, obs_np):
        obs_t = torch.tensor(obs_np, dtype=torch.uint8).to(DEVICE)
        return self.forward(obs_t).item()

    def learn(self, obs_t, returns):
        loss = F.mse_loss(self.forward(obs_t), returns)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.parameters(), 0.5)
        self.optimizer.step()
        return loss.item()


print("ContinuousActor and ContinuousCritic defined.")

## 3. PPO Agent

Same PPO-Clip loop as FrozenLake but adapted for:
- **Continuous actions** (Gaussian policy, tanh squashing)  
- **Pixel observations** stored in the rollout buffer  
- **GAE** (Generalised Advantage Estimation) for lower-variance advantages

In [ ]:
class CarPPOAgent:
    """
    PPO-Clip agent for continuous-action pixel environments.

    Parameters
    ----------
    lr             : Adam learning rate
    gamma          : discount factor
    gae_lambda     : GAE smoothing (0 = TD, 1 = MC)
    epsilon        : PPO clip range
    entropy_coef   : entropy bonus weight
    k_epochs       : gradient updates per collected batch
    rollout_length : environment steps per epoch
    """

    def __init__(self, lr=3e-4, gamma=0.99, gae_lambda=0.95,
                 epsilon=0.2, entropy_coef=0.01, k_epochs=4,
                 rollout_length=512):
        self.actor    = ContinuousActor(N_ACTIONS, lr).to(DEVICE)
        self.critic   = ContinuousCritic(lr).to(DEVICE)
        self.gamma    = gamma
        self.lam      = gae_lambda
        self.epsilon  = epsilon
        self.ent_coef = entropy_coef
        self.k_epochs = k_epochs
        self.rollout_length = rollout_length

    # ------------------------------------------------------------------

    def _collect(self, env, obs):
        """Gather rollout_length transitions. Returns buffer and final obs."""
        buf = []
        for _ in range(self.rollout_length):
            action, log_prob = self.actor.select_action(obs)
            value            = self.critic.value(obs)
            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            buf.append((obs, action, float(reward), done, log_prob, value))
            obs = env.reset()[0] if done else next_obs
        return buf, obs

    def _gae(self, rewards, dones, values):
        """Generalised Advantage Estimation."""
        advantages, gae = [], 0.0
        for r, d, v in zip(reversed(rewards), reversed(dones), reversed(values)):
            delta = r + self.gamma * v * (1 - d) - v
            gae   = delta + self.gamma * self.lam * (1 - d) * gae
            advantages.insert(0, gae)
        returns = [a + v for a, v in zip(advantages, values)]
        return (torch.tensor(advantages, dtype=torch.float32),
                torch.tensor(returns,    dtype=torch.float32))

    # ------------------------------------------------------------------

    def train(self, env, n_updates=500, seed=42):
        """Run the training loop. Returns history dict."""
        obs, _ = env.reset(seed=seed)
        history = {"actor_loss": [], "critic_loss": [], "ep_return": []}
        ep_return, ep_returns = 0.0, []

        for update in range(1, n_updates + 1):
            buf, obs = self._collect(env, obs)

            obss      = np.array([b[0] for b in buf], dtype=np.uint8)
            actions   = np.array([b[1] for b in buf], dtype=np.float32)
            rewards   = [b[2] for b in buf]
            dones     = [b[3] for b in buf]
            log_probs = torch.stack([b[4] for b in buf]).to(DEVICE)
            values    = [b[5] for b in buf]

            # Episode returns for logging
            for r, d in zip(rewards, dones):
                ep_return += r
                if d:
                    ep_returns.append(ep_return)
                    ep_return = 0.0
            history["ep_return"].append(
                np.mean(ep_returns[-10:]) if ep_returns else 0.0)

            advantages, returns = self._gae(rewards, dones, values)
            advantages = ((advantages - advantages.mean())
                          / (advantages.std() + 1e-8)).to(DEVICE)
            returns = returns.to(DEVICE)

            obs_t  = torch.tensor(obss,    dtype=torch.uint8).to(DEVICE)
            act_t  = torch.tensor(actions, dtype=torch.float32).to(DEVICE)

            a_losses, c_losses = [], []
            for _ in range(self.k_epochs):
                a_losses.append(
                    self.actor.learn(obs_t, act_t, log_probs, advantages,
                                     self.epsilon, self.ent_coef))
                c_losses.append(self.critic.learn(obs_t, returns))

            history["actor_loss"].append(sum(a_losses)  / self.k_epochs)
            history["critic_loss"].append(sum(c_losses) / self.k_epochs)

            if update % 50 == 0:
                print(f"Update {update:4d}/{n_updates}"
                      f"  ep_ret={history['ep_return'][-1]:7.1f}"
                      f"  a_loss={history['actor_loss'][-1]:8.4f}"
                      f"  c_loss={history['critic_loss'][-1]:8.4f}")

        env.close()
        return history

    @staticmethod
    def plot(history):
        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        fig.suptitle("PPO Training — CarRacing-v3", fontsize=14, fontweight="bold")
        panels = [
            (axes[0], "ep_return",   "Episode Return (10-ep avg)", "Return",     "#2ca02c"),
            (axes[1], "critic_loss", "Critic Loss (MSE)",          "Loss",       "#d62728"),
            (axes[2], "actor_loss",  "Actor Loss (PPO-Clip)",      "Loss",       "#1f77b4"),
        ]
        for ax, key, title, ylabel, color in panels:
            ax.plot(history[key], color=color, linewidth=2)
            ax.set_title(title, fontsize=11, fontweight="bold")
            ax.set_xlabel("Update Step")
            ax.set_ylabel(ylabel)
            ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()


print("CarPPOAgent defined.")

## 4. Train

| Hyperparameter | Value |
|----------------|-------|
| `rollout_length` | 512 |
| `k_epochs` | 4 |
| `lr` | 3e-4 |
| `gamma` | 0.99 |
| `gae_lambda` | 0.95 |
| `epsilon` | 0.2 |
| `entropy_coef` | 0.01 |

In [ ]:
env = gym.make("CarRacing-v3", render_mode=None, continuous=True)

agent = CarPPOAgent(
    lr=3e-4, gamma=0.99, gae_lambda=0.95,
    epsilon=0.2, entropy_coef=0.01,
    k_epochs=4, rollout_length=512,
)

history = agent.train(env, n_updates=500, seed=42)
CarPPOAgent.plot(history)